# Biomarker Forge — Phase 1: Exploratory Data Analysis

This notebook walks through the processed cohort data and produces:
- Cohort demographics summary
- Outcome distribution
- Biomarker value distributions (signal vs noise)
- Correlation heatmap
- Cohort comparison plots (responders vs non-responders)

All plots are saved to `reports/`.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yaml
from pathlib import Path

reports_dir = Path('../reports')
reports_dir.mkdir(exist_ok=True)

with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

OUTCOME_COL = cfg['cohort']['outcome_column']
ID_COL = cfg['cohort']['id_column']
print('Config loaded')

In [ ]:
# Load clean data
clean_path = Path('../data/synthetic/cohort_clean.csv')
df = pd.read_csv(clean_path)
print(f'Shape: {df.shape}')
df.head(3)

## 1. Cohort Summary

In [ ]:
total = len(df)
responders = df[OUTCOME_COL].sum()
non_responders = total - responders

print(f'Total patients:   {total}')
print(f'Responders:       {responders} ({responders/total*100:.1f}%)')
print(f'Non-responders:   {non_responders} ({non_responders/total*100:.1f}%)')
print()
print('Demographics:')
df.groupby(OUTCOME_COL)[['age','bmi']].describe().round(2)

## 2. Outcome Distribution

In [ ]:
outcome_counts = df[OUTCOME_COL].value_counts().reset_index()
outcome_counts.columns = ['Response', 'Count']
outcome_counts['Response'] = outcome_counts['Response'].map({0: 'Non-Responder', 1: 'Responder'})

fig = px.pie(
    outcome_counts, values='Count', names='Response',
    title='Cohort Outcome Distribution',
    color_discrete_sequence=['#636EFA', '#EF553B'],
    hole=0.4
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.write_image(str(reports_dir / 'outcome_distribution.png'))
fig.show()
print('Saved: reports/outcome_distribution.png')

## 3. Demographics by Outcome

In [ ]:
df_plot = df.copy()
df_plot['Outcome'] = df_plot[OUTCOME_COL].map({0: 'Non-Responder', 1: 'Responder'})

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Age Distribution by Outcome', 'BMI Distribution by Outcome']
)

for outcome_label, color in [('Non-Responder', '#636EFA'), ('Responder', '#EF553B')]:
    subset = df_plot[df_plot['Outcome'] == outcome_label]
    fig.add_trace(
        go.Box(y=subset['age'], name=outcome_label, marker_color=color, showlegend=True),
        row=1, col=1
    )
    fig.add_trace(
        go.Box(y=subset['bmi'], name=outcome_label, marker_color=color, showlegend=False),
        row=1, col=2
    )

fig.update_layout(title_text='Demographics by Outcome', height=400)
fig.write_image(str(reports_dir / 'demographics_by_outcome.png'))
fig.show()
print('Saved: reports/demographics_by_outcome.png')

## 4. Biomarker Distributions — Signal vs Noise

In [ ]:
bm_cols = [c for c in df.columns if c.startswith('BM_')]
n_signal = cfg['cohort']['n_signal_biomarkers']
signal_cols = bm_cols[:n_signal]
noise_cols = bm_cols[n_signal:n_signal+4]  # show 4 noise examples

# Show first 4 signal biomarkers
fig = make_subplots(
    rows=2, cols=4,
    subplot_titles=[f'{c} (signal)' for c in signal_cols[:4]] +
                   [f'{c} (noise)' for c in noise_cols]
)

all_show = [(c, True) for c in signal_cols[:4]] + [(c, False) for c in noise_cols]
for idx, (col, is_signal) in enumerate(all_show):
    row, col_pos = divmod(idx, 4)
    for outcome_val, label, color in [(0, 'Non-R', '#636EFA'), (1, 'Resp', '#EF553B')]:
        subset = df[df[OUTCOME_COL] == outcome_val][col]
        fig.add_trace(
            go.Histogram(x=subset, name=label, opacity=0.6,
                        marker_color=color, showlegend=(idx == 0)),
            row=row+1, col=col_pos+1
        )

fig.update_layout(
    title_text='Biomarker Distributions: Signal (top) vs Noise (bottom)',
    height=550, barmode='overlay'
)
fig.write_image(str(reports_dir / 'biomarker_distributions.png'))
fig.show()
print('Saved: reports/biomarker_distributions.png')

## 5. Correlation Heatmap (Signal Biomarkers)

In [ ]:
corr_cols = signal_cols + [OUTCOME_COL]
corr_matrix = df[corr_cols].corr().round(3)

fig = go.Figure(go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.index,
    colorscale='RdBu',
    zmid=0,
    text=corr_matrix.values.round(2),
    texttemplate='%{text}',
    textfont_size=10,
))
fig.update_layout(
    title='Signal Biomarker Correlation Matrix (incl. Outcome)',
    height=550, width=650
)
fig.write_image(str(reports_dir / 'correlation_heatmap.png'))
fig.show()
print('Saved: reports/correlation_heatmap.png')

## Summary

Phase 1 EDA complete. Outputs written to `reports/`:
- `outcome_distribution.png`
- `demographics_by_outcome.png`
- `biomarker_distributions.png`
- `correlation_heatmap.png`

**Key observations:**
- Signal biomarkers (BM_000–BM_007) show visible distribution separation between responders and non-responders
- Noise biomarkers show overlapping distributions — expected
- Moderate correlations exist among some signal biomarkers (signal injection shares outcome label)

**Next step:** Phase 2 — Biomarker Candidate Selection (univariate stats + FDR correction)